### We will use chroma db for store embedding vector data

In [7]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_community.vectorstores import Chroma

In [8]:
# Sample docs data

sample_data = [
    """
    Artificial intelligence (AI) is the field of building computer systems that can perform tasks such as recognizing patterns, understanding language, and making predictions. Machine learning is a branch of AI where models learn patterns from examples instead of following only hand-written rules. Deep learning uses multi-layer neural networks to learn complex patterns from data.
    """,
    """
    A good foundation for learning AI includes Python programming, basic statistics, probability, and linear algebra. Practice with variables, functions, lists, dictionaries, and data libraries such as NumPy and pandas. Build intuition by experimenting with small datasets and simple models.
    """,
    """
    Machine learning includes supervised and unsupervised learning. Supervised learning uses labeled examples: regression predicts numeric values, while classification predicts categories. Unsupervised learning looks for structure in data without provided labels, such as grouping similar examples with clustering. Split data into training and test sets to measure performance on examples the model has not seen.
    """,
    """
    A neural network learns by adjusting its weights to reduce errors on training examples. Transformers use attention to relate parts of an input sequence. Large language models generate text based on learned patterns, but their answers can be inaccurate or unsupported, so important claims should be checked against reliable sources.
    """,
    """
    Retrieval-augmented generation (RAG) combines document search with a language model. Documents are divided into chunks and converted into vector embeddings. When a user asks a question, the system retrieves relevant chunks and provides them as context to the model. Good source documents and relevant retrieval improve answers, but RAG does not guarantee correctness; showing sources and testing questions with missing answers are important.
    """,
    """
    Responsible AI practice includes protecting personal information, checking for bias, testing on realistic examples, and explaining system limitations. Use data only when you have permission. For high-impact decisions, keep qualified people involved rather than relying on an AI model alone. Treat generated content as a draft that needs verification, not as automatically verified fact.
    """
]

sample_data

['\n    Artificial intelligence (AI) is the field of building computer systems that can perform tasks such as recognizing patterns, understanding language, and making predictions. Machine learning is a branch of AI where models learn patterns from examples instead of following only hand-written rules. Deep learning uses multi-layer neural networks to learn complex patterns from data.\n    ',
 '\n    A good foundation for learning AI includes Python programming, basic statistics, probability, and linear algebra. Practice with variables, functions, lists, dictionaries, and data libraries such as NumPy and pandas. Build intuition by experimenting with small datasets and simple models.\n    ',
 '\n    Machine learning includes supervised and unsupervised learning. Supervised learning uses labeled examples: regression predicts numeric values, while classification predicts categories. Unsupervised learning looks for structure in data without provided labels, such as grouping similar examples

In [9]:
# import tempfile

# temp_dir = tempfile.mkdtemp()

# for i, doc in enumerate(sample_data):
#     with open(f"{temp_dir}_doc_{i}.txt", "w") as fs:
#         fs.write(doc)

# print(f"Temp Dir: {temp_dir}")

### Load the text

In [10]:
for i, doc in enumerate(sample_data):
    with open(f"data/doc_{i}.txt", "w") as fs:
        fs.write(doc)


In [11]:
from langchain_community.document_loaders import (DirectoryLoader, TextLoader)


documents_loader = DirectoryLoader(
    "./data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)
documents = documents_loader.load()
print(f"Documents Loaded: {documents}")

Documents Loaded: [Document(metadata={'source': 'data/doc_3.txt'}, page_content='\n    A neural network learns by adjusting its weights to reduce errors on training examples. Transformers use attention to relate parts of an input sequence. Large language models generate text based on learned patterns, but their answers can be inaccurate or unsupported, so important claims should be checked against reliable sources.\n    '), Document(metadata={'source': 'data/doc_2.txt'}, page_content='\n    Machine learning includes supervised and unsupervised learning. Supervised learning uses labeled examples: regression predicts numeric values, while classification predicts categories. Unsupervised learning looks for structure in data without provided labels, such as grouping similar examples with clustering. Split data into training and test sets to measure performance on examples the model has not seen.\n    '), Document(metadata={'source': 'data/doc_0.txt'}, page_content='\n    Artificial intelli

### Now We will split the documents

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

split_data = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=100,
    chunk_overlap=25,
    length_function=len
)

chunks = split_data.split_documents(documents)

print(f"Chunks: {chunks}")
print(f"\nChunk size: {len(chunks)} of Documents: {len(documents)}")

Chunks: [Document(metadata={'source': 'data/doc_3.txt'}, page_content='A neural network learns by adjusting its weights to reduce errors on training examples.'), Document(metadata={'source': 'data/doc_3.txt'}, page_content='on training examples. Transformers use attention to relate parts of an input sequence. Large'), Document(metadata={'source': 'data/doc_3.txt'}, page_content='an input sequence. Large language models generate text based on learned patterns, but their answers'), Document(metadata={'source': 'data/doc_3.txt'}, page_content='but their answers can be inaccurate or unsupported, so important claims should be checked against'), Document(metadata={'source': 'data/doc_3.txt'}, page_content='be checked against reliable sources.'), Document(metadata={'source': 'data/doc_2.txt'}, page_content='Machine learning includes supervised and unsupervised learning. Supervised learning uses'), Document(metadata={'source': 'data/doc_2.txt'}, page_content='Supervised learning uses labeled e

### Let's convert into embedding format

In [13]:
sample_text = "Hi Hello how are you ?"
embedd_text = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

embedded_data = embedd_text.embed_query(sample_text)

print(f"Embdedd Data: {embedded_data}")

/Users/ajaysharma/Desktop/AI/RAGAI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1020 23:46:30.061000 59593 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10222.99it/s]


Embdedd Data: [-0.06058099493384361, 0.00846791360527277, 0.07731858640909195, 0.003886463353410363, -0.044404931366443634, -0.13614937663078308, 0.08742891997098923, 0.022854609414935112, -0.10503104329109192, -0.003181561827659607, -0.012553415261209011, -0.01733466610312462, -0.012010877020657063, -0.04361359030008316, 0.07386836409568787, -0.03691596910357475, 0.11702334135770798, -0.04535076394677162, -0.038554802536964417, -0.0011414282489567995, -0.08965586870908737, -0.019902681931853294, -0.05289873108267784, 0.008749361149966717, 0.007871183566749096, -0.01776430569589138, 0.03554709255695343, -0.040900107473134995, 0.02913893572986126, -0.08074267953634262, 0.007576161064207554, -0.007862422615289688, 0.1330445408821106, -0.03735477477312088, -0.05135328695178032, 0.0898473709821701, 0.08668643236160278, -0.03968556597828865, -0.04313889890909195, 0.007674474269151688, 0.00755691435188055, 0.01551642082631588, -0.03992359712719917, 0.011193960905075073, -0.01334101241081953,

In [19]:
persistent_db = "./chromadb"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(),
    persist_directory=persistent_db,
    collection_name="rag_collection"
)

print(f"Vector {vector_store}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16717.07it/s]


Vector <langchain_community.vectorstores.chroma.Chroma object at 0x331265a90>


### Similarity Search

In [20]:
query = "what is RAG"

similar_documents = vector_store.similarity_search(query, k=2)
print(f"Length of Results: {len(similar_documents)}")
print(f"Similar results: {similar_documents[0].page_content}")

Length of Results: 2
Similar results: improve answers, but RAG does not guarantee correctness; showing sources and testing questions with


In [21]:
from dotenv import load_dotenv
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# load_dotenv(override=True)

# llm = HuggingFaceEndpoint(
#     repo_id="deepseek-ai/DeepSeek-V4.1-Flash-1017",
#     task="text-generation",
#     max_new_tokens=256,
# )
# model = ChatHuggingFace(llm=llm)

# print(f"{model.invoke("what is highest score in ODI cricket")}")


from langchain_openai import ChatOpenAI
import os
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=os.environ.get("OPENAI_API_KEY")
)

result = llm.invoke("What is highest score in cricket ODI")
print(f"Result: {result}")


Result: content='The highest team score in a One Day International (ODI) cricket match is 481/6 by England against Australia in Nottingham on June 19, 2018. The highest individual score in a ODI match is 264 runs by Rohit Sharma (India) against Sri Lanka on November 13, 2014.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 15, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Eb8gFrlahI7t54KmZYlOHq0KRaGR0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a15a87-c62c-7732-adf9-8603783a2e9c-0' tool_calls=[] invalid_too

In [22]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate


In [24]:
retriever = vector_store.as_retriever(
    kwargs={"k":3}
)

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x331265a90>, search_kwargs={})

In [26]:
system_prompt = """ You are genious please provide relavant anser based on relavant context, if you dont know say i dont know
    Context: {context}
    """


In [29]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

ValueError: Unexpected message type: System. Use one of 'human', 'user', 'ai', 'assistant', or 'system'.

In [ ]:
documents_chain = create_stuff_documents_chain(
    llm,
    prompt
)

documents_chain

NameError: name 'prompt' is not defined